# Conhecendo o Spark: o que é e por que o time escolheu

O time escolheu o Spark porque o projeto da Arena Analytics vai crescer. Este notebook reúne os conceitos que
explicam essa escolha.

## 1. O que é processar dados

**Processar dados** é aplicar uma sequência de passos a dados brutos até chegar a um resultado útil. Exemplo do
projeto, gols por Copa:

1. **Ler** cada partida do arquivo
2. **Calcular** os gols da partida (gols do mandante + gols do visitante)
3. **Agrupar** as partidas pelo ano da Copa
4. **Somar** os gols de cada grupo

Em SQL: `SELECT ano, SUM(gols) FROM partidas GROUP BY ano`.

O mesmo cálculo com o **pandas**, uma biblioteca do Python para manipular dados em tabelas:

In [1]:
import pandas as pd

partidas = pd.read_csv("../../data/raw/WorldCupMatches.csv")
partidas = partidas.dropna(subset=["Year"])          # o arquivo tem linhas vazias no fim
partidas = partidas.astype({"Year": "int", "Home Team Goals": "int", "Away Team Goals": "int"})

partidas["gols"] = partidas["Home Team Goals"] + partidas["Away Team Goals"]
gols_por_copa = partidas.groupby("Year")["gols"].sum()

gols_por_copa.head()

Year
1930     70
1934     70
1938     92
1950     88
1954    172
Name: gols, dtype: int64

Funcionou, e foi instantâneo.

## 2. O que aconteceu dentro do computador

Essas poucas linhas de pandas usaram três partes do computador, cada uma com um papel:

| Parte | Papel |
|---|---|
| **Disco** | onde o dado fica guardado de forma permanente, mesmo com o computador desligado |
| **Memória (RAM)** | onde o dado fica enquanto está sendo trabalhado; é muito mais rápida que o disco, mas menor e temporária |
| **Processador (CPU)** | quem faz as contas; cada **núcleo** faz uma coisa por vez |

O caminho do dado foi este:

1. O arquivo `WorldCupMatches.csv` estava **no disco**.
2. O `read_csv` **copiou o arquivo inteiro para a memória**. O DataFrame `partidas` é essa cópia, carregada de uma vez.
3. O **processador** fez as contas (somar os gols, agrupar por ano) em cima do que estava na memória. O processador
   só trabalha com o que está na memória: tudo o que está no disco precisa ser carregado antes.
4. O resultado, `gols_por_copa`, também ficou na memória. Para guardar, seria o caminho inverso: gravar da memória
   para o disco.

E se o arquivo não estivesse no seu computador, mas num servidor ou na nuvem? Antes de chegar à memória, ele teria
que atravessar a **rede**, que é ainda mais lenta que o disco.

## 3. O limite de uma máquina só

Com o histórico das Copas, o dado coube na memória e a conta levou menos de um segundo. Mas cada parte da máquina
tem um limite:

- **Memória:** o dado inteiro precisa caber nela, e o pandas costuma usar várias vezes o tamanho do arquivo, porque
  cria cópias a cada passo. Um notebook comum tem de 8 a 32 GB. Quando o dado não cabe, o programa para com erro de
  memória, ou fica muito lento, porque o computador passa a usar o disco como memória de reserva.
- **Processador:** o pandas faz as contas basicamente com um núcleo. Mesmo que a máquina tenha vários, os outros
  ficam parados.
- **Disco e rede:** antes de qualquer conta, o dado precisa sair de onde está guardado e chegar à memória. Ler um
  arquivo enorme do disco, ou baixá-lo pela rede, leva tempo, mesmo que a conta em si seja rápida.

Agora pense no futuro da Arena Analytics. Além das partidas, o fornecedor também manda um **feed de eventos** (a
pasta `events`): cada passe, falta, chute e gol de cada partida, com o minuto e o jogador. Nas Copas mais recentes,
são mais de mil lances por partida. E se um dia a empresa receber dados de rastreamento (a posição dos 22 jogadores
e da bola várias vezes por segundo), cada partida passa a ter milhões de linhas, e uma Copa inteira, centenas de
milhões.

**O problema não é o código, é a máquina.** Com esse volume, o dado não cabe na memória, um núcleo não dá conta das
contas, e só trazer o dado até a memória já leva muito tempo.

## 4. Duas saídas: uma máquina maior ou mais máquinas

| Estratégia | Como funciona | Limite |
|---|---|---|
| **Escalar na vertical** | trocar por uma máquina maior, com mais memória, mais núcleos e disco mais rápido | tem teto: em algum momento não existe máquina maior, e cada passo custa bem mais caro |
| **Escalar na horizontal** | juntar várias máquinas trabalhando juntas | é preciso dividir o trabalho e juntar os resultados |

Na horizontal, as máquinas **somam** os seus recursos. Dez máquinas com 16 GB têm 160 GB de memória juntas, os
núcleos de todas trabalham ao mesmo tempo, e cada uma lê do disco ou da rede só a sua parte. Nenhuma precisa guardar
o dado inteiro.

A segunda saída é a que o Spark usa.

## 5. Processamento distribuído

Dividir um cálculo entre várias máquinas é o **processamento distribuído**. Veja o nosso cálculo de gols por Copa
feito por três computadores:

```
                     arquivo de partidas
               /              |              \
         pedaço 1         pedaço 2         pedaço 3
       computador A     computador B     computador C

       1930: 25 gols    1930: 21 gols    1930: 24 gols     <- cada um soma só o seu pedaço
       1950: 30 gols    1950: 31 gols    1950: 27 gols        (resultado parcial)
       ...              ...              ...
               \              |              /
                juntar os parciais de cada Copa
                1930: 25 + 21 + 24 = 70 gols
                1950: 30 + 31 + 27 = 88 gols
```

O trabalho tem três fases:

1. **Dividir:** cada computador recebe uma parte das partidas.
2. **Calcular em paralelo:** cada um carrega a sua parte para a própria memória e soma os gols de cada Copa no seu
   pedaço, sem falar com ninguém. Com três computadores, essa fase leva cerca de um terço do tempo.
3. **Juntar:** os resultados parciais de cada Copa se encontram num lugar só para dar o total. Os parciais trocam de
   máquina pela rede, e isso tem um custo, que volta na aula 4.

## 6. Cluster

Um **cluster** é um grupo de computadores ligados em rede que trabalham como se fossem um só. Cada computador do
grupo é chamado de **nó**. Na prática, esses computadores costumam estar na nuvem (AWS, Azure, Google Cloud) ou
num data center da empresa, e não na mesa de ninguém.

Montar o processamento distribuído "na mão", num cluster, seria um pesadelo. Alguém teria que decidir:

- como dividir o arquivo e qual computador fica com qual pedaço;
- como enviar o código para cada computador;
- como juntar os parciais no final;
- o que fazer se um computador quebrar no meio do trabalho.

É aqui que entra o Spark.

## 7. Então, o que é o Apache Spark?

O **Apache Spark** é um motor de processamento distribuído, de código aberto. Você escreve a lógica uma vez, como
se o dado fosse uma tabela só (ler, calcular, agrupar, somar), e o Spark cuida do resto:

- divide o dado em pedaços, chamados **partições**;
- distribui o trabalho entre os computadores do cluster;
- executa os pedaços em paralelo;
- junta os resultados;
- refaz automaticamente o pedaço de um computador que falhou.

O Spark nasceu em 2009 na Universidade da Califórnia em Berkeley e hoje é mantido pela Apache Software Foundation.
Ele pode ser usado em Scala, Java, SQL e Python. Neste curso usamos o **PySpark**, a versão em Python.

O caminho do dado é o mesmo do pandas (disco ou rede, depois memória, depois processador), com uma diferença
importante: **o Spark não carrega o dado inteiro de uma vez.** Ele divide o dado em pedaços, as **partições**. Cada
máquina carrega só os pedaços que recebeu, processa um de cada vez em cada núcleo e libera a memória para o próximo.
Por isso o Spark consegue processar dados maiores que a memória de uma máquina e até maiores que a memória de todas
as máquinas juntas.

Além de distribuir o trabalho, o Spark procura manter os resultados intermediários na **memória RAM** em vez de
gravá-los no disco a cada passo, como faziam ferramentas mais antigas (como o Hadoop MapReduce). Em cálculos com
muitos passos, isso faz uma diferença grande de tempo.

## 8. Quando o Spark faz sentido

O Spark não é a melhor escolha para todo dado. Compare com o pandas, que você usou no começo deste notebook:

| | pandas | Spark |
|---|---|---|
| Quantas máquinas | uma | uma ou muitas |
| Como carrega o dado | o arquivo inteiro na memória | em partições, um pedaço por vez em cada núcleo |
| Quantos núcleos calculam | basicamente um | todos os disponíveis |
| Se o dado não cabe na memória | erro ou lentidão extrema | segue processando em partes |
| Bom para | dado que cabe na memória de uma máquina e vai continuar cabendo | dado que não cabe numa máquina, ou que vai crescer até não caber |
| Custo de começar | quase nenhum: importa e calcula | alguns segundos para ligar e tempo coordenando as partes |

Em dado pequeno, o custo de começar do Spark aparece, e o pandas termina antes. A pergunta que decide é: **o dado
cabe numa máquina hoje, e vai continuar cabendo?**

E a Arena Analytics? Hoje o fornecedor manda alguns arquivos pequenos de histórico. Vamos medir:

In [2]:
# liste os CSVs de ../../data/raw com o tamanho
from pathlib import Path

for arquivo in sorted(Path("../../data/raw").glob("*.csv")):
    print(f"{arquivo.name:<26} {arquivo.stat().st_size / 1024:>8.1f} KB")

WorldCupMatches.csv           163.1 KB
WorldCupPlayers.csv          1699.3 KB
WorldCups.csv                   1.6 KB


Poucos KB. Se o projeto fosse só isso, o pandas resolveria, como vimos no começo do vídeo.

Mas o projeto da Arena Analytics não é só isso. A cada Copa entram mais seleções e mais jogos (a partir de 2026 são
104 partidas por edição), e o feed de eventos multiplica o volume por centenas. É por isso
que o time escolheu o Spark: o pipeline é escrito uma vez e continua funcionando quando o dado crescer.

## 9. E no nosso computador?

Se o Spark existe para dividir o trabalho entre vários computadores, como ele vai rodar aqui, só com o seu?

O Spark tem um **modo local**: ele roda inteiro numa máquina só e usa os **núcleos** do processador como se fossem as
máquinas do cluster: cada núcleo processa uma partição por vez, com a memória dessa máquina. Um computador com 8 núcleos vira, para o Spark, 8 "trabalhadores" processando pedaços do
dado ao mesmo tempo. É paralelismo de verdade, só que dentro de uma máquina. Foi isso que pedimos na sessão do vídeo
1.2, com o `master("local[*]")`.

```
num cluster (produção)                        no seu computador (modo local)
+-----------+ +-----------+ +-----------+     +------------------------------------------+
| máquina A | | máquina B | | máquina C |     | seu computador                           |
| pedaço 1  | | pedaço 2  | | pedaço 3  |     | núcleo 1   núcleo 2   núcleo 3   ...     |
+-----------+ +-----------+ +-----------+     | pedaço 1   pedaço 2   pedaço 3           |
                                              +------------------------------------------+
```

Três pontos para não confundir:

1. **O código é o mesmo.** O que muda entre o modo local e um cluster é a configuração da sessão, não a lógica que
   você escreve.
2. **No modo local, o limite é o do seu computador.** O Spark não ganha memória nem processador extra: ele usa melhor
   o que a máquina já tem. Por isso o modo local é onde o time desenvolve e testa o pipeline.
3. **Em produção, o cluster.** Quando o pipeline da Arena Analytics processar o feed de eventos de todas as Copas,
   ele vai rodar num cluster na nuvem (serviços como Databricks, Amazon EMR e Google Dataproc entregam isso pronto). O
   código que você escrever aqui é o que vai para lá.

Quantos "trabalhadores" o seu container tem?

In [3]:
import os
os.cpu_count()

12

## Por que o time escolheu o Spark

- O histórico de hoje é pequeno, mas o feed de eventos e as próximas Copas vão multiplicar o volume.
- Com o Spark, o pipeline é escrito uma vez: roda no modo local durante o desenvolvimento e num cluster quando o
  volume pedir.
- Desenvolver com o dado de hoje é rápido: cada célula roda em segundos.